# Transform motif count tables For pseudo-bacodes

## Make a pseudo-barcodes allele reference table, Using THP1 mono but the table can be used for brain and gut

In [ ]:
import numpy as np
import pandas as pd
df_brain = pd.read_csv("read_counts_R1R2/THP1_RNA_matched_barcodes.csv",index_col=0)
RNA_columns=['Naive_THP1mono_ZC111_R', 'Naive_THP1mono_ZC112_R',
       'Naive_THP1mono_ZC113_R', 'Naive_THP1mono_ZC37_R',
       'Naive_THP1mono_ZC60_R', 'Naive_THP1mono_ZC90_R']

def process_column_name(name):
    # Removing characters that differentiate the columns (like R and R2 in this case)
    return name.replace('R-', '-').replace('R2-', '-')

grouped_df= df_brain.iloc[(986):,:][RNA_columns].groupby(process_column_name, axis=1).sum()
grouped_df = pd.merge(grouped_df,df_index,left_index=True, right_index=True)
grouped_df=distribute_values_order(grouped_df)

pseudo_barcodes_df = sum_barcode_groups(grouped_df ).set_index("Pseudo-barcode").sort_index()
enhancer_id = grouped_df[["enhancer_id","Pseudo-barcode"]].drop_duplicates("Pseudo-barcode").set_index("Pseudo-barcode")
pseudo_barcodes_df = pd.merge(pseudo_barcodes_df,enhancer_id,left_index=True, right_index=True)

###################################################################################################################################
#make allele_barcode table and produce reshaped count table
import pandas as pd
sorted_dict = pd.read_csv("indexing/motifdisrupt_grouped.csv")
count_table = pseudo_barcodes_df 
count_table=count_table[(count_table.index.str.contains('motifdisrupt')|count_table.index.str.contains('alt')|count_table.index.str.contains('ref'))]
count_table=count_table[~count_table.index.str.contains('cg')]
# Adding 'Barcode' column to count_table
count_table['Barcode'] = count_table.groupby('enhancer_id').cumcount() + 1

# Initialize allele_type list
allele_type = []

# Iterate over 'enhancer_id' in count_table
for i in count_table['enhancer_id']:
    if sorted_dict.apply(lambda column: i in column.values).any():
        column_name = sorted_dict.apply(lambda column: i in column.values).idxmax()
        allele_type.append(column_name)
    else:
        allele_type.append("no motif")

# Add 'allele_type' to count_table
count_table['allele_type'] = allele_type
count_table = count_table[count_table['allele_type'] != 'no motif']
#count_table['allele_barcode'] = count_table.groupby('enhancer_id').cumcount() + 1
count_table['allele_barcode'] = "_"+count_table['allele_type']+"_Barcode_"+count_table['Barcode'].astype(str)

count_table[['allele_type','allele_barcode']].to_csv("indexing/20240609_motifdisrupt_allele_pseudobarcode.csv")

## Convert matched table to motif-allele table for THP1 mono, Brain, and Gut

In [43]:
#Use premade allele barcode table
import pandas as pd
def count_table_motif_transformation(count_table):
    sorted_dict = pd.read_csv("indexing/motifdisrupt_grouped.csv")
    count_table_alllele_barcode = pd.read_csv("indexing/20240609_motifdisrupt_allele_pseudobarcode.csv" ,index_col=0)

    count_table = pd.merge(count_table, count_table_alllele_barcode, left_index=True, right_index=True, how='inner')
    columns = []
    for sample in count_table.columns.drop(['enhancer_id','allele_type', 'allele_barcode']):
        for allele in sorted_dict.columns:
            for i in range(1,6):
                columns.append(sample+f"_{allele}_"+"Barcode_"+str(i))
    rows = sorted_dict['motif'].tolist()
    df = pd.DataFrame(index=rows,columns=columns)
    sample_list = count_table.columns.drop(['enhancer_id','allele_type','allele_barcode'])
    for i in count_table.iterrows():
        enhancer_id = i[1]["enhancer_id"]

        allele_barcode = i[1]['allele_barcode']

        allele_type = i[1]['allele_type']

        if allele_type =="motif":
            motif_enhancer = enhancer_id
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]        
        elif allele_type == "ref":
            motif_enhancer = sorted_dict[sorted_dict['ref'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        elif allele_type == "alt1":
            motif_enhancer = sorted_dict[sorted_dict['alt1'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        elif allele_type == "alt2":
            motif_enhancer = sorted_dict[sorted_dict['alt2'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        elif allele_type == "alt3":
            motif_enhancer = sorted_dict[sorted_dict['alt3'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        df=df.fillna(0)
    return df 


In [ ]:
THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/THP1mono20240412Pseudo_DNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/THP1mono20240412Pseudo_DNA_matched_barcodes_motif.csv")
THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/THP1mono20240412_DNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/THP1mono20240412_DNA_matched_barcodes_motif.csv")
THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/THP1mono20240412_RNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/THP1mono20240412_RNA_matched_barcodes_motif.csv")

THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes_motif.csv")
THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes_motif.csv")

THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/GutR1R2merged20240404_DNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/GutR1R2merged20240404_DNA_matched_barcodes_motif.csv")
THP1mono_pseudobarcodes = pd.read_csv("read_counts_R1R2/GutR1R2merged20240404_RNA_matched_barcodes.csv",index_col=0)
count_table_motif_transformation(THP1mono_pseudobarcodes).to_csv("read_counts_R1R2/GutR1R2merged20240404_RNA_matched_barcodes_motif.csv")

In [42]:
import pandas as pd
def split_by_subtype_make_motif(barcode_file, regions, cell,nucleotide):
    # Read the original DataFrame
    df = pd.read_csv(barcode_file, index_col=0)
    # Loop over each region
    for region in regions:
        # Select columns that contain the region name
        cols = [col for col in df.columns if region in col]+['enhancer_id']
        df_region = df[cols]
        # Sort the columns
        df_region = df_region[sorted(df_region.columns)]
        # Define the output filename
        filename = f"read_counts_R1R2/{cell}_{region}_{nucleotide}_matched_barcodes.csv"
        # Save the DataFrame to a CSV file
        df_region.to_csv(filename)
        count_table_motif_transformation(df_region).to_csv(f"read_counts_R1R2/{cell}_{region}_{nucleotide}_matched_barcodes_motif.csv")

In [44]:
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes.csv", regions = ['Hippocampus', 'Cortex', 'Striatum'], cell='BrainR1R2merged20240404',nucleotide = "DNA")
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes.csv", regions = ['Hippocampus', 'Cortex', 'Striatum'], cell='BrainR1R2merged20240404',nucleotide = "RNA")
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/GutR1R2merged20240404_DNA_matched_barcodes.csv", regions = ['LI','SI'], cell='GutR1R2merged20240404',nucleotide = "DNA")
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/GutR1R2merged20240404_RNA_matched_barcodes.csv", regions = ['LI','SI'], cell='GutR1R2merged20240404',nucleotide = "RNA")

/tmp/ipykernel_30801/50023175.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.fillna(0)
/tmp/ipykernel_30801/50023175.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.fillna(0)
/tmp/ipykernel_30801/50023175.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.fillna(0)
/tmp/ipykernel_30801/50023175.py:48: Future

# Transform motif count tables For regular barcodes

In [ ]:
from collections import defaultdict
import pandas as pd
df_motif_allele=pd.read_csv("indexing/ALT_REF_LookUpTable_filtered_amended_motif_alt_ref_20240127.csv")
# Initialize a defaultdict of sets
grouped_dict = defaultdict(set)

# Iterate over each row in the dataframe
for _, row in df_motif_allele.iterrows():
    grouped_dict[row['motif']].add(row['ref'])
    grouped_dict[row['motif']].add(row['alt'])

# Convert defaultdict to a regular dictionary
grouped_dict = {k: list(v) for k, v in grouped_dict.items()}

print("maximum number of elements that have the same ref enhancer", max(len(v) for v in grouped_dict.values()))

def custom_sort_list(lst):
    return sorted(lst, key=lambda x: (not x.startswith('ref'), x))

# Sorting each list in the dictionary
sorted_dict = {k: custom_sort_list(v) for k, v in grouped_dict.items()}
sorted_dict = pd.DataFrame.from_dict(sorted_dict, orient='index').sort_values([3,2,1,0])
sorted_dict.reset_index(inplace=True)
sorted_dict.columns = ['motif', 'ref', 'alt1','alt2','alt3']
sorted_dict.to_csv("indexing/motifdisrupt_grouped.csv",index=None)

## Build 20240609_motifdisrupt_allele_barcode.csv

In [ ]:

#make allele_barcode table and produce reshaped count table
import pandas as pd
sorted_dict = pd.read_csv("indexing/motifdisrupt_grouped.csv")
count_table = pd.read_csv('read_counts_R1R2/HEK293T_RNA_matched_barcodes.csv' ,index_col=0)
count_table=count_table[(count_table.index.str.contains('motifdisrupt')|count_table.index.str.contains('alt')|count_table.index.str.contains('ref'))]
count_table=count_table[~count_table.index.str.contains('cg')]
# Adding 'Barcode' column to count_table
count_table['Barcode'] = count_table.groupby('enhancer_id').cumcount() + 1

# Initialize allele_type list
allele_type = []

# Iterate over 'enhancer_id' in count_table
for i in count_table['enhancer_id']:
    if sorted_dict.apply(lambda column: i in column.values).any():
        column_name = sorted_dict.apply(lambda column: i in column.values).idxmax()
        allele_type.append(column_name)
    else:
        allele_type.append("no motif")

# Add 'allele_type' to count_table
count_table['allele_type'] = allele_type
count_table = count_table[count_table['allele_type'] != 'no motif']
#count_table['allele_barcode'] = count_table.groupby('enhancer_id').cumcount() + 1
count_table['allele_barcode'] = "_"+count_table['allele_type']+"_Barcode_"+count_table['Barcode'].astype(str)
columns = []
for sample in count_table.columns.drop(['enhancer_id','Barcode', 'allele_type', 'allele_barcode']):
    for allele in sorted_dict.columns:
        for i in range(1,16):
            columns.append(sample+f"_{allele}_"+"Barcode_"+str(i))
rows = sorted_dict['motif'].tolist()
df = pd.DataFrame(index=rows,columns=columns)
sample_list = count_table.columns.drop(['enhancer_id','Barcode','allele_type','allele_barcode'])

for i in count_table.iterrows():
    enhancer_id = i[1]["enhancer_id"]

    allele_barcode = i[1]['allele_barcode']

    allele_type = i[1]['allele_type']

    if allele_type =="motif":
        motif_enhancer = enhancer_id
        for sample in sample_list:
            current_new_column_name = sample+allele_barcode
            df.loc[motif_enhancer,current_new_column_name] = i[1][sample]        
    elif allele_type == "ref":
        motif_enhancer = sorted_dict[sorted_dict['ref'] == enhancer_id]["motif"]
        for sample in sample_list:
            current_new_column_name = sample+allele_barcode
            df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
    elif allele_type == "alt1":
        motif_enhancer = sorted_dict[sorted_dict['alt1'] == enhancer_id]["motif"]
        for sample in sample_list:
            current_new_column_name = sample+allele_barcode
            df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
    elif allele_type == "alt2":
        motif_enhancer = sorted_dict[sorted_dict['alt2'] == enhancer_id]["motif"]
        for sample in sample_list:
            current_new_column_name = sample+allele_barcode
            df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
    elif allele_type == "alt3":
        motif_enhancer = sorted_dict[sorted_dict['alt3'] == enhancer_id]["motif"]
        for sample in sample_list:
            current_new_column_name = sample+allele_barcode
            df.loc[motif_enhancer,current_new_column_name] = i[1][sample]


In [47]:
#Use premade allele barcode table
import pandas as pd
def count_table_motif_transformation(count_table):
    sorted_dict = pd.read_csv("indexing/motifdisrupt_grouped.csv")
    count_table_alllele_barcode = pd.read_csv("indexing/20240609_motifdisrupt_allele_barcode.csv" ,index_col=0)

    count_table = pd.merge(count_table, count_table_alllele_barcode, left_index=True, right_index=True, how='inner')
    columns = []
    for sample in count_table.columns.drop(['enhancer_id','allele_type', 'allele_barcode']):
        for allele in sorted_dict.columns:
            for i in range(1,16):
                columns.append(sample+f"_{allele}_"+"Barcode_"+str(i))
    rows = sorted_dict['motif'].tolist()
    df = pd.DataFrame(index=rows,columns=columns)
    sample_list = count_table.columns.drop(['enhancer_id','allele_type','allele_barcode'])
    for i in count_table.iterrows():
        enhancer_id = i[1]["enhancer_id"]

        allele_barcode = i[1]['allele_barcode']

        allele_type = i[1]['allele_type']

        if allele_type =="motif":
            motif_enhancer = enhancer_id
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]        
        elif allele_type == "ref":
            motif_enhancer = sorted_dict[sorted_dict['ref'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        elif allele_type == "alt1":
            motif_enhancer = sorted_dict[sorted_dict['alt1'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        elif allele_type == "alt2":
            motif_enhancer = sorted_dict[sorted_dict['alt2'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        elif allele_type == "alt3":
            motif_enhancer = sorted_dict[sorted_dict['alt3'] == enhancer_id]["motif"]
            for sample in sample_list:
                current_new_column_name = sample+allele_barcode
                df.loc[motif_enhancer,current_new_column_name] = i[1][sample]
        df=df.fillna(0)
    return df 


In [ ]:

count_table = pd.read_csv('read_counts_R1R2/HEK293T_RNA_matched_barcodes.csv' ,index_col=0)
count_table_motif_transformation(count_table).to_csv("read_counts_R1R2/HEK293T_RNA_matched_barcodes_reshaped_motif.csv")
count_table = pd.read_csv('read_counts_R1R2/HEK293T_DNA_matched_barcodes.csv' ,index_col=0)
count_table_motif_transformation(count_table).to_csv("read_counts_R1R2/HEK293T_DNA_matched_barcodes_reshaped_motif.csv")

count_table = pd.read_csv('read_counts_R1R2/HMC3_RNA_matched_barcodes.csv' ,index_col=0)
count_table_motif_transformation(count_table).to_csv("read_counts_R1R2/HMC3_RNA_matched_barcodes_reshaped_motif.csv")
count_table = pd.read_csv('read_counts_R1R2/HMC3_DNA_matched_barcodes.csv' ,index_col=0)
count_table_motif_transformation(count_table).to_csv("read_counts_R1R2/HMC3_DNA_matched_barcodes_reshaped_motif.csv")

count_table = pd.read_csv('read_counts_R1R2/THP1_RNA_matched_barcodes.csv' ,index_col=0)
count_table_motif_transformation(count_table).to_csv("read_counts_R1R2/THP1_RNA_matched_barcodes_reshaped_motif.csv")
count_table = pd.read_csv('read_counts_R1R2/THP1_DNA_matched_barcodes.csv' ,index_col=0)
count_table_motif_transformation(count_table).to_csv("read_counts_R1R2/THP1_DNA_matched_barcodes_reshaped_motif.csv")

In [48]:
import pandas as pd
import re
def split_by_subtype_make_motif(barcode_file, regions, cell,nucleotide):
    # Read the original DataFrame
    df = pd.read_csv(barcode_file, index_col=0)
    # Loop over each region
    for region in regions:
        # Select columns that contain the region name
        cols = [col for col in df.columns if region in col]+['enhancer_id']
        df_region = df[cols]
        if region == 'IFNG':
            df_region = df_region.loc[:, ~df_region.columns.str.contains('LPS', case=False)]
        # Sort the columns
        df_region = df_region[sorted(df_region.columns)]
        # Define the output filename
        filename = f"read_counts_R1R2/{cell}_{region}_{nucleotide}_matched_barcodes.csv"
        # Save the DataFrame to a CSV file
        df_region.to_csv(filename)
        count_table_motif_transformation(df_region).to_csv(f"read_counts_R1R2/{cell}_{region}_{nucleotide}_matched_barcodes_motif.csv")

In [1]:
import pandas as pd
import re
def split_by_subtype_make_motif(barcode_file, regions, cell,nucleotide):
    '''
    This function skip reprocessing the subtype separately because the THP1 and HMC3 are filtered additionally in 1.3.2
    '''
    # Read the original DataFrame
    df = pd.read_csv(barcode_file, index_col=0)
    # Loop over each region
    for region in regions:
        # Select columns that contain the region name
        cols = [col for col in df.columns if region in col]
        df_region = df[cols]
        if region == 'IFNG':
            df_region = df_region.loc[:, ~df_region.columns.str.contains('LPS', case=False)]
        if region == 'Naive':
            df_region = df_region.loc[:, ~df_region.columns.str.contains('THP1mono', case=False)]
        # Sort the columns
        df_region = df_region[sorted(df_region.columns)]
        # Define the output filename
        filename = f"read_counts_R1R2/{cell}_{region}_{nucleotide}_matched_barcodes_motif.csv"
        # Save the DataFrame to a CSV file
        df_region.to_csv(filename)
#after 1.3.2
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/HMC3_DNA_matched_barcodes_reshaped_motif.csv", regions = ['IFNG','Naive','LPSIFNG','IFNB'], cell='HMC3',nucleotide = "DNA")
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/HMC3_RNA_matched_barcodes_reshaped_motif.csv", regions = ['IFNG','Naive','LPSIFNG','IFNB'], cell='HMC3',nucleotide = "RNA")
#after 1.3.2
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/THP1_DNA_matched_barcodes_reshaped_motif.csv", regions = ['IFNG','THP1mono','Naive','LPSIFNG','IFNB'], cell='THP1',nucleotide = "DNA")
split_by_subtype_make_motif(barcode_file = "read_counts_R1R2/THP1_RNA_matched_barcodes_reshaped_motif.csv", regions = ['IFNG','THP1mono','Naive','LPSIFNG','IFNB'], cell='THP1',nucleotide = "RNA")

In [63]:
import pandas as pd
import re
def split_by_subtype_make_motif(barcode_file, regions, cell):
    '''
    This function skip reprocessing the subtype separately because the THP1 and HMC3 are filtered additionally in 1.3.2
    '''
    # Read the original DataFrame
    df = pd.read_csv(barcode_file, index_col=0)
    # Loop over each region
    for region in regions:
        # Select columns that contain the region name
        indices = [index for index in df.index if region in index]
        df_region = df.loc[indices]
        if region == 'IFNG':
            df_region = df_region.loc[ ~df_region.index.str.contains('LPS', case=False),:]
        # Sort the columns
        #df_region = df_region[sorted(df_region.columns)]
        # Define the output filename
        filename = f"annotation_barcodes/mpra3_annot_{cell}_{region}_barcodes_motif.csv"
        # Save the DataFrame to a CSV file
        df_region.to_csv(filename)

#after 1.3.2
split_by_subtype_make_motif(barcode_file = "annotation_barcodes/mpra3_annot_THP1Macrophage_barcodes_motif.csv", regions = ['IFNG','THP1mono','Naive','LPSIFNG','IFNB'], cell='THP1')
split_by_subtype_make_motif(barcode_file = 'annotation_barcodes/mpra3_annot_HMC3_barcodes_motif.csv', regions = ['IFNG','Naive','LPSIFNG','IFNB'], cell='HMC3')


# Make ALT-REF allele only tables.

In [2]:
import os
import pandas as pd
for f in os.listdir("read_counts_R1R2/"):
    if f.endswith('matched_barcodes_reshaped.csv'):
        df = pd.read_csv(f"read_counts_R1R2/{f}",index_col=0)
        df[df.index.str.contains('alt')].to_csv(f"read_counts_R1R2/allele_only/{f.replace('matched_barcodes_reshaped','matched_barcodes_reshaped_allele')}")